# FEATURE ENGINEERING

> - We'll prepare features for weekly purchased-unit demand across online & offline channels combined, covering all categories.
> - Each forecast will cover the next 13 weeks together, using information available after a completed week.

# FORECAST SETUP & TARGET TABLE

> - We'll load the formatted dataset saved from EDA as df & build one total per Monday–Sunday week.
> - We'll exclude the two incomplete boundary weeks & keep recorded purchased units, including returned transactions.

In [6]:
# Loading the formatted transaction dataset saved from EDA.
import pandas as pd
from IPython.display import display

processed_path = "../data/processed/cleaned_data.csv"
df = pd.read_csv(processed_path, parse_dates=["purchase_date"])

print(f"Transactions: {len(df)} | Columns: {df.shape[1]}")
print(f"Date range: {df['purchase_date'].min():%Y-%m-%d} to {df['purchase_date'].max():%Y-%m-%d}")
print(f"Dates with records: {df['purchase_date'].nunique()}")
print(f"Channels: {sorted(df['online/offline'].unique())}")
display(df.head(3))

Transactions: 10000 | Columns: 26
Date range: 2023-01-01 to 2024-12-30
Dates with records: 730
Channels: ['Offline', 'Online']


,transaction_id,customer_id,purchase_date,age,gender,location,online/offline,online_store,category,item_purchased,...,festival/sale,shipping,delivery_speed,delivery_time,subscription_status,payment_method,review_rating,return_status,previous_purchases,frequency_of_purchases
0,500000,1440,2023-07-22,52,Other,Indore,Offline,In-Store Purchase,Clothing,Jeans,...,Standard Day,0,N/A (Offline),0,No,Credit Card,5,Not Returned,2,Quarterly
1,500001,2326,2023-01-16,19,Female,Bhubaneswar,Online,Ajio,Clothing,Jacket,...,Standard Day,0,Express,1,No,Debit Card,4,Returned,3,Quarterly
2,500002,2680,2023-01-24,40,Male,Bangalore,Online,Myntra,Clothing,Saree,...,Standard Day,0,Standard,7,Yes,Cash on Delivery,4,Not Returned,13,Monthly


In [9]:
# Building combined-channel totals for complete Monday–Sunday weeks.
weekly_all = df.groupby(pd.Grouper(
    key="purchase_date", freq="W-SUN")).agg(
    weekly_units=("quantity", "sum"),
    recorded_days=("purchase_date", "nunique")
).reset_index().rename(columns={"purchase_date": "week_end"})

partial_weeks = weekly_all[weekly_all["recorded_days"] != 7].copy()
display(partial_weeks)

# Keeping complete weeks and the required columns.
weekly_target = weekly_all[weekly_all["recorded_days"] == 7]
weekly_target = weekly_target[["week_end", "weekly_units"]]
weekly_target = weekly_target.reset_index(drop=True)

print(f"Complete weeks: {len(weekly_target)}")
display(weekly_target.head())

,week_end,weekly_units,recorded_days
0,2023-01-01,14,1
105,2025-01-05,18,1


Complete weeks: 104


,week_end,weekly_units
0,2023-01-08,138
1,2023-01-15,147
2,2023-01-22,129
3,2023-01-29,136
4,2023-02-05,148


# TIME SPLIT & AVAILABLE INFORMATION

> - We'll use train_test_split with shuffle off to keep the weekly data in date order.
> - We'll reserve the last 13 weeks for test & the preceding 13 weeks for validation, matching our approximately quarterly planning window.
> - Each forecast will use purchases known by its completed-week cutoff. The next 13 weeks' actual purchases will remain unavailable as inputs.
> - EDA covered the whole dataset, so the later evaluation will be retrospective.

In [12]:
# Splitting complete weeks in date order using the forecast horizon.
from sklearn.model_selection import train_test_split

horizon_weeks = 13
weekly_target = weekly_target.sort_values("week_end").reset_index(drop=True)

train_validation, test_period = train_test_split(
weekly_target, test_size=horizon_weeks, shuffle=False)

train_period, validation_period = train_test_split(
train_validation, test_size=horizon_weeks, shuffle=False)

validation_origin = train_period["week_end"].max()
test_origin = validation_period["week_end"].max()
latest_origin = test_period["week_end"].max()

print(f"Weeks - train: {len(train_period)}, validation: {len(validation_period)}, test: {len(test_period)}")
print(f"Validation cutoff: {validation_origin.date()}")
print(f"Test cutoff: {test_origin.date()}")
print(f"Latest cutoff: {latest_origin.date()}")

Weeks - train: 78, validation: 13, test: 13
Validation cutoff: 2024-06-30
Test cutoff: 2024-09-29
Latest cutoff: 2024-12-29


# CALENDAR & PAST PURCHASE FEATURES

> - We'll use forecast distance, forecast month, the latest completed week's units & a four-week mean.
> - Each row will represent one future week forecast from one completed-week cutoff.
> - The history features will stay fixed across all 13 weeks forecast from that cutoff. These candidates will be assessed later during modeling.

In [14]:
# Preparing the latest week's units & the mean of four completed weeks.
history_weeks = 4

origin_features = weekly_target.copy()
origin_features = origin_features.rename(columns={
"week_end": "origin_week_end","weekly_units": "units_latest_week"})
origin_features["mean_units_4weeks"] = (
origin_features["units_latest_week"].rolling(history_weeks).mean())

display(origin_features.head(5))

,origin_week_end,units_latest_week,mean_units_4weeks
0,2023-01-08,138,NaN
1,2023-01-15,147,NaN
2,2023-01-22,129,NaN
3,2023-01-29,136,137.5
4,2023-02-05,148,140.0


In [18]:
# Preparing weekly dates and matching future purchases.
number_of_weeks = len(weekly_target)

week_dates = pd.date_range(start=weekly_target["week_end"].min(),
periods=number_of_weeks + horizon_weeks,freq="W-SUN")

units = weekly_target["weekly_units"].tolist() + [None] * horizon_weeks

horizon_tables = []

for week_ahead in range(1, horizon_weeks + 1):
    table = origin_features.copy()
    table["horizon_weeks"] = week_ahead
    table["forecast_week_end"] = week_dates[week_ahead:week_ahead + number_of_weeks]
    table["target_units"] = units[week_ahead:week_ahead + number_of_weeks]
    horizon_tables.append(table)

feature_table = pd.concat(horizon_tables, ignore_index=True)
feature_table["forecast_month"] = feature_table["forecast_week_end"].dt.month
feature_table = feature_table.sort_values(
    ["origin_week_end", "horizon_weeks"]).reset_index(drop=True)

display(feature_table.head(4))

,origin_week_end,units_latest_week,mean_units_4weeks,horizon_weeks,forecast_week_end,target_units,forecast_month
0,2023-01-08,138,NaN,1,2023-01-15,147.0,1
1,2023-01-08,138,NaN,2,2023-01-22,129.0,1
2,2023-01-08,138,NaN,3,2023-01-29,136.0,1
3,2023-01-08,138,NaN,4,2023-02-05,148.0,2


# FINAL CHECKS & MODELING HANDOFF

> - We'll remove only the first three cutoffs without four weeks of history & inspect missing values and repeated forecast rows.
> - We'll keep training targets before validation begins & retain one complete 13-week block for validation and test.
> - We'll prepare x, y & date objects, plus future feature rows. Model fitting & evaluation will come later.

In [20]:
# Removing the initial cutoffs without enough history & printing checks.
feature_columns = ["horizon_weeks", "forecast_month",
"units_latest_week", "mean_units_4weeks"]

print(f"Cutoffs without four-week history: {origin_features['mean_units_4weeks'].isna().sum()}")

first_ready_week = weekly_target["week_end"].iloc[history_weeks - 1]
feature_table = feature_table[feature_table["origin_week_end"] >= first_ready_week].copy()

duplicate_rows = feature_table.duplicated(["origin_week_end", "horizon_weeks"]).sum()
print(f"Feature rows: {len(feature_table):,}")
print(f"Repeated cutoff & horizon pairs: {duplicate_rows}")
print("Missing values in the four input features:")
display(feature_table[feature_columns].isna().sum())

Cutoffs without four-week history: 3
Feature rows: 1,313
Repeated cutoff & horizon pairs: 0
Missing values in the four input features:


horizon_weeks        0
forecast_month       0
units_latest_week    0
mean_units_4weeks    0
dtype: int64

In [21]:
# Selecting training rows & one 13-week block at each later cutoff.
train_table = feature_table[feature_table["forecast_week_end"] <= validation_origin].copy()
validation_table = feature_table[feature_table["origin_week_end"] == validation_origin].copy()
test_table = feature_table[feature_table["origin_week_end"] == test_origin].copy()
future_table = feature_table[feature_table["origin_week_end"] == latest_origin].copy()

split_summary = pd.DataFrame({"split": ["Training", "Validation", "Test", "Future"],
"rows": [len(train_table), len(validation_table),len(test_table), len(future_table)],
"missing_targets": [train_table["target_units"].isna().sum(),
validation_table["target_units"].isna().sum(),test_table["target_units"].isna().sum(),
future_table["target_units"].isna().sum()]})

display(split_summary)
display(validation_table.head(3))

,split,rows,missing_targets
0,Training,884,0
1,Validation,13,0
2,Test,13,0
3,Future,13,13


,origin_week_end,units_latest_week,mean_units_4weeks,horizon_weeks,forecast_week_end,target_units,forecast_month
1001,2024-06-30,143,143.25,1,2024-07-07,117.0,7
1002,2024-06-30,143,143.25,2,2024-07-14,158.0,7
1003,2024-06-30,143,143.25,3,2024-07-21,131.0,7


In [22]:
# Preparing matching feature, target & date objects for the next notebook.
date_columns = ["origin_week_end", "forecast_week_end"]

x_train = train_table[feature_columns].copy()
y_train = train_table["target_units"].copy()
dates_train = train_table[date_columns].copy()

x_validation = validation_table[feature_columns].copy()
y_validation = validation_table["target_units"].copy()
dates_validation = validation_table[date_columns].copy()

x_test = test_table[feature_columns].copy()
y_test = test_table["target_units"].copy()
dates_test = test_table[date_columns].copy()

x_future = future_table[feature_columns].copy()
dates_future = future_table[date_columns].copy()

print(f"Training x & y: {x_train.shape}, {y_train.shape}")
print(f"Validation x & y: {x_validation.shape}, {y_validation.shape}")
print(f"Test x & y: {x_test.shape}, {y_test.shape}")
print(f"Future x: {x_future.shape}")

display(dates_future)

Training x & y: (884, 4), (884,)
Validation x & y: (13, 4), (13,)
Test x & y: (13, 4), (13,)
Future x: (13, 4)


,origin_week_end,forecast_week_end
1339,2024-12-29,2025-01-05
1340,2024-12-29,2025-01-12
1341,2024-12-29,2025-01-19
1342,2024-12-29,2025-01-26
1343,2024-12-29,2025-02-02
1344,2024-12-29,2025-02-09
1345,2024-12-29,2025-02-16
1346,2024-12-29,2025-02-23
1347,2024-12-29,2025-03-02
1348,2024-12-29,2025-03-09


In [23]:
# Saving weekly totals & prepared features for modeling.
weekly_target.to_csv("../data/processed/weekly_target.csv", index=False)
feature_table.to_csv("../data/processed/features.csv", index=False)

# FEATURE ENGINEERING SUMMARY

> - We prepared weekly purchased-unit demand across online & offline channels for a forecast window of 13 weeks.
> - Our four candidate features describe forecast distance, forecast month, latest-week purchases & the mean of four completed weeks.
> - History remains fixed at each forecast cutoff, keeping purchases from the future window out of the inputs.
> - We prepared 884 training examples, 13 validation examples, 13 test examples & 13 future feature rows.
> - Model fitting, validation across multiple periods & final evaluation will follow in the modeling notebook.